QC purge: remove flagged ANs from a subgroup and find what must be re-aligned

Use this every time new ANs are added to a subgroup's `qc_excluded_ANs.tsv`:

1. Add the ANs to `<group>/<subgroup>/qc_excluded_ANs.tsv`
   (columns: AN, reason, flagged_in, date_added)
2. Set `PURGE_TAG` below, run **Setup** and **Step 1 (snapshot)**
3. Re-run, in order:
   - `01_get_data`: the "filtering subgroups" cell and the "Get gff3" cell
   - `02_gene_pairs`: all cells
   - `prank_tree`: all cells
4. Run **Step 2 (compare)** and review the table
5. Run **Step 3 (move stale files)**, first with `DRY_RUN = True`, then `False`
6. Re-run `03a_run_prank` and `03b_run_clustalo` with `RE_RUN = False`
   (they only align the pairs whose output was moved away), then `04_aln_analysis`

Everything this notebook saves goes to `<group>/<subgroup>/qc_purge/<PURGE_TAG>/`,
and one summary line per purge is appended to `qc_purge_log.tsv`.

In [ ]:
import hashlib
import shutil
from pathlib import Path

import polars as pl
from ete4 import Tree

In [ ]:
CWD = Path().resolve()
BASE = CWD

print(f"Folder where groups (data) folder should be: {BASE}")

In [ ]:
groups_map = {
    "plants_plt": ["solanaceae"],
    "plants_mit": ["solanaceae"],
}

EXCLUDE_FILE = "qc_excluded_ANs.tsv"  # one per subgroup: <group>/<subgroup>/qc_excluded_ANs.tsv
LOG_FILE = BASE / "qc_purge_log.tsv"
PURGE_TAG = "2026-10-07"  # one folder per purge: use a new tag for every new purge

# alignment runs to fix -> does that aligner use the gene-pair tree?
RUNS = {
    "prank_2026-09-30": True,
    "clustalo_2026-09-30": False,
}

# Pruning can write the same tree with children in a different order. PRANK gets the
# same guide tree, so by default those pairs are not re-aligned.
RERUN_REORDERED_TREES = False

In [ ]:
def md5(path: Path) -> str | None:
    return hashlib.md5(path.read_bytes()).hexdigest() if path.exists() else None


def fasta_ids(path: Path) -> list[str]:
    with open(path) as fh:
        return [line[1:].split()[0] for line in fh if line.startswith(">")]


def tree_canon_md5(path: Path) -> str | None:
    """MD5 of the tree with children sorted, so a mere reordering hashes the same."""
    if not path.exists():
        return None
    tree = Tree(path.read_text(), parser=5)
    for node in tree.traverse("postorder"):
        if node.children:
            node.children.sort(key=lambda c: min(c.leaf_names()))
    return hashlib.md5(tree.write(parser=5).encode()).hexdigest()


def input_hashes(subg_dir: Path, gene_pairs) -> pl.DataFrame:
    """MD5 of each gene pair's PRANK/Clustal inputs (FASTA and guide tree)."""
    rows = []
    for gp in gene_pairs:
        tree_file = subg_dir / "gene_pair_trees" / f"gp_{gp}.nwk"
        rows.append(
            {
                "gene_pair": gp,
                "fasta_md5": md5(subg_dir / "gene_pair_fastas" / f"gp_{gp}.fasta"),
                "tree_md5": md5(tree_file),
                "tree_canon_md5": tree_canon_md5(tree_file),
            }
        )
    schema = {c: pl.String for c in ["gene_pair", "fasta_md5", "tree_md5", "tree_canon_md5"]}
    return pl.DataFrame(rows, schema=schema)


def move(src: Path, dest_dir: Path) -> None:
    if DRY_RUN:
        print(f"    would move {src.relative_to(BASE)} -> {dest_dir.relative_to(BASE)}/")
        return
    dest_dir.mkdir(parents=True, exist_ok=True)
    shutil.move(src, dest_dir / src.name)


def read_excluded(g: str, subg: str) -> set[str]:
    """ANs in <group>/<subgroup>/qc_excluded_ANs.tsv (empty set if there is no file)."""
    f = BASE / g / subg / EXCLUDE_FILE
    if not f.exists():
        return set()
    ex = pl.read_csv(f, separator="\t")
    if "AN" not in ex.columns:
        raise ValueError(f"{f} needs an 'AN' column")
    dups = ex.filter(pl.col("AN").is_duplicated())["AN"].unique().to_list()
    if dups:
        print(f"  warning: listed more than once in {EXCLUDE_FILE}: {dups}")
    return set(ex["AN"])

## Step 1: snapshot (run BEFORE re-running 01 / 02 / prank_tree)

In [ ]:
for g, subgs in groups_map.items():
    for subg in subgs:
        subg_dir = BASE / g / subg
        out = subg_dir / "qc_purge" / PURGE_TAG
        print(f"{g}/{subg}")

        if out.exists():
            print(f"  snapshot {out.relative_to(BASE)} already exists, not overwriting")
            print("  (delete that folder by hand if you really want a new snapshot)")
            continue

        excluded_ans = read_excluded(g, subg)
        print(f"  {EXCLUDE_FILE}: {len(excluded_ans)} ANs")
        # catch typos / version mismatches (.1 vs .2): every AN should be in the group TSV
        group_ans = set(pl.read_csv(BASE / g / f"{g}.tsv", separator="\t")["AN"])
        unknown = sorted(excluded_ans - group_ans)
        if unknown:
            print(f"  warning: not in {g}.tsv (typo or version?): {unknown}")

        tsv = pl.read_csv(subg_dir / f"{subg}.tsv", separator="\t")
        pending = tsv.filter(pl.col("AN").is_in(list(excluded_ans)))
        if pending.height == 0:
            print("  no excluded AN in this subgroup, nothing to purge")
            continue

        print(f"  ANs that will be removed ({pending.height}):")
        for an, taxid in pending.select("AN", "ncbi_taxid").iter_rows():
            print(f"    {an} (taxid {taxid})")

        # preview what 01 will do: another AN of the same taxid takes over, or the taxon leaves
        lineage_file = BASE / g / f"{g}_raw_lineage.tsv"
        if lineage_file.exists():
            group_tsv = pl.read_csv(lineage_file, separator="\t")
            for an, taxid in pending.select("AN", "ncbi_taxid").iter_rows():
                alt = (
                    group_tsv.filter(
                        pl.col("ncbi_taxid") == taxid,
                        ~pl.col("AN").is_in(list(excluded_ans)),
                    )
                    .sort("Genome_length", descending=True)
                    .head(1)
                )
                if alt.height:
                    print(f"    taxid {taxid}: {an} will be replaced by {alt['AN'][0]}")
                else:
                    print(f"    taxid {taxid}: no other AN, taxon leaves the subgroup")

        atlas = pl.read_csv(subg_dir / f"{subg}_summary_atlas.tsv", separator="\t")
        gp_df = pl.read_csv(subg_dir / "gene_pairs_filtered.tsv", separator="\t")
        hit = atlas.filter(pl.col("AN").is_in(pending["AN"].to_list()))
        hit_gp = set(hit["gene_pair"]) & set(gp_df["gene_pair"])
        print(
            f"  {hit.height} IGRs from these ANs, in {len(hit_gp)}/{gp_df.height} "
            "filtered gene pairs (trees may change for more, see Step 2)"
        )

        out.mkdir(parents=True)
        shutil.copy(subg_dir / EXCLUDE_FILE, out / "qc_excluded_ANs_used.tsv")
        pending.write_csv(out / "removed_ANs.tsv", separator="\t")
        shutil.copy(subg_dir / f"{subg}.tsv", out / "subgroup_before.tsv")
        shutil.copy(subg_dir / f"{subg}_summary_atlas.tsv", out / "atlas_before.tsv")
        shutil.copy(subg_dir / "gene_pairs_filtered.tsv", out / "gene_pairs_filtered_before.tsv")
        input_hashes(subg_dir, gp_df["gene_pair"]).write_csv(
            out / "inputs_before.tsv", separator="\t"
        )
        print(f"  snapshot -> {out.relative_to(BASE)}")

print("\nNext: re-run 01 (filtering subgroups + gff3 cells), 02 (all), sandbox/prank_tree/ncbi_tree.ipynb (all).")

## Step 2: compare (run AFTER re-running 01 / 02 / prank_tree)

In [ ]:
for g, subgs in groups_map.items():
    for subg in subgs:
        subg_dir = BASE / g / subg
        out = subg_dir / "qc_purge" / PURGE_TAG
        print(f"{g}/{subg}")

        if not (out / "inputs_before.tsv").exists():
            print("  no snapshot for this purge, skipping")
            continue
        snap_time = (out / "inputs_before.tsv").stat().st_mtime
        excluded_ans = read_excluded(g, subg)

        tsv_file = subg_dir / f"{subg}.tsv"
        atlas_file = subg_dir / f"{subg}_summary_atlas.tsv"
        gp_file = subg_dir / "gene_pairs_filtered.tsv"
        gp_now = pl.read_csv(gp_file, separator="\t")
        gp_before = pl.read_csv(out / "gene_pairs_filtered_before.tsv", separator="\t")
        pairs_now = gp_now["gene_pair"].to_list()

        # --- were the upstream notebooks re-run after the snapshot?
        problems = []
        for f, nb in [(tsv_file, "01"), (atlas_file, "01"), (gp_file, "02")]:
            if f.stat().st_mtime < snap_time:
                problems.append(f"{f.name} is older than the snapshot: re-run {nb}")
        for folder, ext, nb in [
            ("gene_pair_fastas", "fasta", "02"),
            ("gene_pair_trees", "nwk", "prank_tree"),
        ]:
            files = [subg_dir / folder / f"gp_{gp}.{ext}" for gp in pairs_now]
            old = [f for f in files if f.exists() and f.stat().st_mtime < snap_time]
            if old:
                problems.append(f"{len(old)} files in {folder} older than the snapshot: re-run {nb}")

        # --- is every excluded AN really gone?
        tsv_now = pl.read_csv(tsv_file, separator="\t")
        atlas_now = pl.read_csv(atlas_file, separator="\t")
        left = sorted(excluded_ans & set(tsv_now["AN"]))
        if left:
            problems.append(f"still in {tsv_file.name}: {left} (is the exclusion in 01?)")
        left = sorted(excluded_ans & set(atlas_now["AN"]))
        if left:
            problems.append(f"still in {atlas_file.name}: {left}")

        atlas_before = pl.read_csv(out / "atlas_before.tsv", separator="\t")
        bad_ids = set(
            atlas_before.filter(pl.col("AN").is_in(list(excluded_ans)))["ID"].cast(pl.String)
        )
        leftover = {}
        for gp in pairs_now:
            fasta = subg_dir / "gene_pair_fastas" / f"gp_{gp}.fasta"
            if fasta.exists():
                ids = bad_ids.intersection(fasta_ids(fasta))
                if ids:
                    leftover[gp] = sorted(ids)
        if leftover:
            problems.append(f"excluded IGRs still in {len(leftover)} FASTAs: {dict(list(leftover.items())[:3])}")

        if problems:
            print("  STOP, fix these first:")
            for p in problems:
                print(f"    {p}")
            continue

        # --- what happened to the ANs
        ans_before = set(pl.read_csv(out / "subgroup_before.tsv", separator="\t")["AN"])
        ans_now = set(tsv_now["AN"])
        print(f"  ANs removed: {sorted(ans_before - ans_now)}")
        print(f"  ANs added (replacements): {sorted(ans_now - ans_before)}")

        # --- classify every gene pair by comparing its inputs
        before = {r["gene_pair"]: r for r in pl.read_csv(out / "inputs_before.tsv", separator="\t").iter_rows(named=True)}
        now = {r["gene_pair"]: r for r in input_hashes(subg_dir, pairs_now).iter_rows(named=True)}
        igr_before = dict(gp_before.select("gene_pair", "igr_count").iter_rows())
        igr_now = dict(gp_now.select("gene_pair", "igr_count").iter_rows())

        rows = []
        for gp in sorted(before.keys() | now.keys()):
            b, n = before.get(gp), now.get(gp)
            if n is None:
                status = "dropped"
            elif n["fasta_md5"] is None:
                status = "missing_fasta"
            elif b is None:
                status = "new"
            elif n["fasta_md5"] != b["fasta_md5"]:
                status = "fasta_changed"
            elif n["tree_canon_md5"] != b["tree_canon_md5"]:
                status = "tree_changed"
            elif n["tree_md5"] != b["tree_md5"]:
                status = "tree_reordered"
            else:
                status = "unchanged"
            rows.append(
                {
                    "gene_pair": gp,
                    "status": status,
                    "igr_count_before": igr_before.get(gp),
                    "igr_count_now": igr_now.get(gp),
                    "missing_tree": n is not None and n["tree_md5"] is None,
                }
            )
        changes = pl.DataFrame(rows)
        changes.write_csv(out / "changes.tsv", separator="\t")

        counts = dict(changes.group_by("status").len().iter_rows())
        print("  gene pairs: " + ", ".join(f"{k}={v}" for k, v in sorted(counts.items())))
        if counts.get("tree_changed", 0) > 0.5 * counts.get("unchanged", 0):
            print(
                "  note: many trees changed without their FASTA changing; the subgroup tree "
                "was probably rescaled (removed taxon at the root), so PRANK re-runs widely"
            )
        no_tree = changes.filter(pl.col("missing_tree"))["gene_pair"].to_list()
        if no_tree:
            print(f"  warning: no tree for {no_tree}, PRANK will report 'missing tree'")
        if counts.get("missing_fasta"):
            print("  warning: some filtered pairs have no FASTA, check 02's output")

        # --- re-run lists per alignment run
        log_row = {
            "purge_tag": PURGE_TAG,
            "group": g,
            "subgroup": subg,
            "ANs_removed": ";".join(sorted(ans_before - ans_now)),
            "ANs_added": ";".join(sorted(ans_now - ans_before)),
            **{
                s: counts.get(s, 0)
                for s in ["unchanged", "fasta_changed", "tree_changed", "tree_reordered", "new", "dropped"]
            },
        }
        tree_redo = ["tree_changed"] + (["tree_reordered"] if RERUN_REORDERED_TREES else [])
        for run, uses_tree in RUNS.items():
            redo = ["new", "fasta_changed"] + (tree_redo if uses_tree else [])
            rerun = changes.filter(pl.col("status").is_in(redo))["gene_pair"].to_list()
            (out / f"rerun_{run}.txt").write_text("".join(f"{gp}\n" for gp in rerun))
            log_row[f"rerun_{run}"] = len(rerun)
            print(f"  {run}: {len(rerun)} pairs to align -> rerun_{run}.txt")

        # --- one line per purge in the project log (replaced if Step 2 is run again)
        log = pl.DataFrame([log_row])
        if LOG_FILE.exists():
            old = pl.read_csv(LOG_FILE, separator="\t", infer_schema_length=None)
            old = old.filter(
                ~((pl.col("purge_tag") == PURGE_TAG) & (pl.col("group") == g) & (pl.col("subgroup") == subg))
            )
            log = pl.concat([old, log], how="diagonal_relaxed")
        log.write_csv(LOG_FILE, separator="\t")

## Step 3: move stale files (DRY_RUN = True first)
Alignments that must be redone go to `alignments/<run>/_stale/<PURGE_TAG>/`, so 03a/03b
(RE_RUN = False) align exactly those pairs. FASTAs/trees of dropped pairs and GFF3s of
removed ANs go to `qc_purge/<PURGE_TAG>/orphans/`. Nothing is deleted.

In [ ]:
DRY_RUN = False  # Step 3 only prints what it would move

In [ ]:
for g, subgs in groups_map.items():
    for subg in subgs:
        subg_dir = BASE / g / subg
        out = subg_dir / "qc_purge" / PURGE_TAG
        print(f"{g}/{subg}{' [DRY RUN]' if DRY_RUN else ''}")
        moved = "would move" if DRY_RUN else "moved"

        if not (out / "changes.tsv").exists():
            print("  no changes.tsv (run Step 2 first), skipping")
            continue
        changes = pl.read_csv(out / "changes.tsv", separator="\t")
        dropped = changes.filter(pl.col("status") == "dropped")["gene_pair"].to_list()

        # --- alignments
        for run in RUNS:
            run_dir = subg_dir / "alignments" / run
            if not run_dir.exists():
                print(f"  {run}: no run folder, skipping")
                continue
            rerun = (out / f"rerun_{run}.txt").read_text().split()
            stale_dir = run_dir / "_stale" / PURGE_TAG
            n = 0
            for gp in rerun + dropped:
                f = run_dir / f"gp_{gp}_aligned.fasta"
                if f.exists():
                    move(f, stale_dir)
                    n += 1
            print(f"  {run}: {moved} {n} alignments to _stale/{PURGE_TAG}/")

        # --- inputs that no longer belong to a filtered pair
        keep = set(pl.read_csv(subg_dir / "gene_pairs_filtered.tsv", separator="\t")["gene_pair"])
        for folder, ext in [("gene_pair_fastas", ".fasta"), ("gene_pair_trees", ".nwk")]:
            orphans = [
                f for f in sorted((subg_dir / folder).glob(f"gp_*{ext}"))
                if f.name[3 : -len(ext)] not in keep
            ]
            for f in orphans:
                move(f, out / "orphans" / folder)
            print(f"  {folder}: {moved} {len(orphans)} orphan files")

        ans = set(pl.read_csv(subg_dir / f"{subg}.tsv", separator="\t")["AN"])
        gff3_orphans = [f for f in sorted((subg_dir / "gff3").glob("*.gff3")) if f.stem not in ans]
        for f in gff3_orphans:
            move(f, out / "orphans" / "gff3")
        print(f"  gff3: {moved} {len(gff3_orphans)} orphan files")

if DRY_RUN:
    print("\nDRY RUN: nothing moved. Set DRY_RUN = False and run this cell again.")
else:
    print("\nNext: 03a_run_prank and 03b_run_clustalo with RE_RUN = False, then 04_aln_analysis.")